<a href="https://colab.research.google.com/github/icosahedron31/Facial-Expression-Recognition/blob/main/VisionTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle wandb onnx -Uq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 23.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! mkdir ~/.kaggle

In [ ]:
! mkdir -p ~/.kaggle && echo KGAT_8cb0c83c63cdaf3612ac74923b3aa519 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
!cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:03<00:00, 96.6MB/s]



In [ ]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tdola23 (tdola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

**Imports**

In [ ]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cuda


**Read and split dataset**

In [ ]:
df = pd.read_csv('../content/train.csv')
train_ds, test_ds = train_test_split(df, test_size = 0.3, random_state=42)
test_ds, val_ds = train_test_split(test_ds, test_size = 0.5, random_state=42)
train_ds.head(), val_ds.head()

(       emotion                                             pixels
 14421        0  157 154 157 137 94 99 115 104 120 110 128 98 1...
 9226         4  11 13 17 20 20 21 24 29 28 28 30 33 40 50 57 6...
 994          2  242 250 154 76 80 69 57 53 50 59 55 45 46 49 5...
 28675        0  111 111 112 110 111 111 109 106 99 88 44 68 12...
 4600         3  7 1 4 17 9 11 25 30 33 31 23 38 29 42 56 48 60...,
        emotion                                             pixels
 17147        3  136 136 137 136 138 137 137 135 133 102 52 36 ...
 14040        3  45 26 33 65 47 36 37 35 36 66 88 69 53 38 47 3...
 21306        3  80 78 78 77 76 76 80 115 132 111 93 80 86 92 8...
 22365        3  250 250 250 251 251 251 252 159 58 69 50 44 42...
 22196        1  255 255 255 255 253 255 165 85 62 57 63 72 90 ...)

**Transform**

In [ ]:
from PIL import Image
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05)
    ),
    transforms.ToTensor(),
    transforms.Resize((48, 48)),
    transforms.Normalize([0.5], [0.5])
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((48, 48)),
    transforms.Normalize([0.5], [0.5])
])

**Creating Dataset**

In [ ]:
class CustomImageDataset(Dataset):
  def __init__(self, df, transform=None) :
    self.df = df
    self.transform = transform
  def __getitem__(self, index) :
    rows = self.df.iloc[index]
    pixels = rows["pixels"]
    label = torch.tensor(rows["emotion"], dtype=torch.long).to(device)
    pixels = np.fromstring(pixels, dtype=np.uint8, sep=" ").reshape(48, 48)
    pixels = self.transform(pixels).to(device)
    return pixels, label
  def __len__(self) :
    return len(self.df)

In [ ]:
train_ds = CustomImageDataset(train_ds, transform)
val_ds = CustomImageDataset(val_ds, val_transform)

**Model**

In [ ]:
import torch
import torch.nn as nn

class ViTSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_extractor = nn.Conv2d(in_channels = 1, out_channels = 512, kernel_size = 3, stride = 3)
        self.layer = nn.TransformerEncoderLayer(
            d_model = 512,
            dropout=0.0,
            nhead = 1,
            dim_feedforward=1024,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)
        self.pool = nn
        self.linear = nn.Linear(512, 7)
    def forward(self, x):

        x = self.patch_extractor(x) #B, E, H, W
        x = x.flatten(2) #B, E, H * W
        x = x.transpose(1, 2) #B, H * W, E
       # print(x.shape)
        x = self.transformer_encoder(x)
       # print(x.shape)
        x = x.mean(dim = 1)
        x = self.linear(x)

       # print(x.shape)
        return x

In [ ]:
import torch
import torch.nn as nn

class ViTPositional(nn.Module):
    def __init__(self, n_embd = 256, num_layers = 3, dropout=0.3, n_head = 1):
        super().__init__()
        self.n_embd = n_embd
        self.num_layers = 3,
        self.dropout = dropout
        self.n_head = n_head
        self.patch_extractor = nn.Conv2d(in_channels = 1, out_channels = self.n_embd, kernel_size = 3, stride = 3)
        self.layer = nn.TransformerEncoderLayer(
            d_model = self.n_embd,
            dropout= self.dropout,
            nhead = self.n_head,
            dim_feedforward=512,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)
        self.pool = nn
        self.linear = nn.Linear(self.n_embd, 7)
        self.pos = nn.Parameter(torch.randn(1,  256, self.n_embd) * 0.2)
    def forward(self, x):

        x = self.patch_extractor(x) #B, E, H, W

        x = x.flatten(2) #B, E, H * W
        x = x.transpose(1, 2) #B, H * W, E
        x = x + self.pos;
       # print(x.shape)
        x = self.transformer_encoder(x)
       # print(x.shape)
        x = x.mean(dim = 1)
        x = self.linear(x)

       # print(x.shape)
        return x

**Training Loop**

In [ ]:
import copy
def train_loop(train_ds, val_ds, model_name, learning_rate, batch_size, model, EPOCHS=30,
               weight_decay=None) :

  train_dataloader = DataLoader(train_ds, batch_size, shuffle=True)
  val_dataloader = DataLoader(val_ds, batch_size, shuffle=False)
  criterion = nn.CrossEntropyLoss()
  best_model = model
  best_acc_val = 0
  if weight_decay :
    optimizer = Adam(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  else :
    optimizer = Adam(model.parameters(), lr = learning_rate)
  for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_loss_val = 0
    total_acc_val = 0
    model.train()
    for inputs, labels in train_dataloader:
      optimizer.zero_grad()
      outputs = model(inputs).to(device)
      train_loss = criterion(outputs, labels)
      total_loss_train += train_loss.item()
      train_loss.backward()

      train_acc = (torch.argmax(outputs, axis = 1) == labels).sum().item()
      total_acc_train += train_acc
      optimizer.step()
    model.eval()
    with torch.no_grad():
      for inputs, labels in val_dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        val_loss = criterion(outputs, labels)
        total_loss_val += val_loss.item()

        val_acc = (torch.argmax(outputs, axis = 1) == labels).sum().item()
        total_acc_val += val_acc
    if total_acc_val > best_acc_val :
        best_acc_val = total_acc_val
        best_model = copy.deepcopy(model.state_dict())
        if epoch > 10 :
          torch.save(best_model, f"{model_name}.pt")

    print(f'''Epoch {epoch+1}/{EPOCHS}, Train Loss: {round(total_loss_train/train_ds.__len__(), 4)} Train Accuracy {round((total_acc_train)/train_ds.__len__() * 100, 4)}
                Validation Loss: {round(total_loss_val/val_ds.__len__(), 4)} Validation Accuracy: {round((total_acc_val)/val_ds.__len__() * 100, 4)}''')


    wandb.log({
        "epoch": epoch,
        "train_loss": total_loss_train / train_ds.__len__(),
        "train_acc": round((total_acc_train)/train_ds.__len__() * 100, 4),
        "val_loss": round(total_loss_val/val_ds.__len__(), 4),
        "val_acc": round((total_acc_val)/val_ds.__len__() * 100, 4)
    })

  return best_model

**Training**

In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = ViTSmall().to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="overfitting_small_ds_simple_numlayer3",
    config = {
    "project": "ml_hw_04",
    "model": "SimpleViT",
    "input": {
        "subset_size": 100,
        "dataset": "train_ds"
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "overfit_small_dataset",
            "group": "ViT"
        }
    }
)
model = train_loop(small_ds, small_ds, "ViT", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


/tmp/ipykernel_1390/4041787203.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


epoch,▁▂▃▅▆▇█
train_acc,▁▃▅▆▆▇█
train_loss,█▅▄▃▂▂▁
val_acc,▁▂▇▆█▄▆
val_loss,█▇▃▂▁▄▄
epoch,6
train_acc,28.3887
train_loss,0.05452
val_acc,26.7936
val_loss,0.0555


Epoch 1/100, Train Loss: 0.2903 Train Accuracy 18.0
                Validation Loss: 0.2645 Validation Accuracy: 20.0
Epoch 2/100, Train Loss: 0.2557 Train Accuracy 19.0
                Validation Loss: 0.2276 Validation Accuracy: 25.0
Epoch 3/100, Train Loss: 0.2437 Train Accuracy 17.0
                Validation Loss: 0.2286 Validation Accuracy: 31.0
Epoch 4/100, Train Loss: 0.2374 Train Accuracy 21.0
                Validation Loss: 0.2232 Validation Accuracy: 27.0
Epoch 5/100, Train Loss: 0.2317 Train Accuracy 26.0
                Validation Loss: 0.2204 Validation Accuracy: 29.0
Epoch 6/100, Train Loss: 0.2237 Train Accuracy 26.0
                Validation Loss: 0.2205 Validation Accuracy: 27.0
Epoch 7/100, Train Loss: 0.2267 Train Accuracy 24.0
                Validation Loss: 0.2179 Validation Accuracy: 30.0
Epoch 8/100, Train Loss: 0.2226 Train Accuracy 25.0
                Validation Loss: 0.2161 Validation Accuracy: 30.0
Epoch 9/100, Train Loss: 0.2353 Train Accuracy 23.0
    

epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇███
train_acc,▁▁▁▂▂▂▂▁▂▂▂▂▃▃▂▃▃▄▄▅▄▅▆▄▅▆▆▆▆▇██▇▆▆▇▇███
train_loss,██▇▇█▇▇▇▇▇▆▇▆▆▆▅▆▅▄▄▆▄▃▃▃▂▂▂▂▂▂▁▃▄▂▂▂▁▁▁
val_acc,▁▁▂▁▁▂▂▃▃▃▃▃▄▄▄▅▅▆▆▆▄▅▆▇▆█▇█████▆▇▇█████
val_loss,██████▇▇▇██▇▇▇▇▆▆▆▅▅▃▆▆▄▄▃▂▃▃▃▁▁▁▁▃▂▂▂▁▁
epoch,99
train_acc,100
train_loss,0.00144
val_acc,100
val_loss,0.0013


In [ ]:
from torch.utils.data import Subset

model = ViTSmall().to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="simple_numlayer3",
    config = {
    "project": "ml_hw_04",
    "model": "SimpleTransformer",

    "training": {
        "epochs": 30,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "ViT"
        }
    }
)
model = train_loop(train_ds, val_ds, "cnn_transformer", 0.0001, 32, model, 30)
torch.save(model, "small_vit.pt")
wandb.finish()


/tmp/ipykernel_1390/4041787203.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


Epoch 1/30, Train Loss: 0.0567 Train Accuracy 23.965
                Validation Loss: 0.0562 Validation Accuracy: 26.0274
Epoch 2/30, Train Loss: 0.0559 Train Accuracy 25.3035
                Validation Loss: 0.0564 Validation Accuracy: 21.5463
Epoch 3/30, Train Loss: 0.0555 Train Accuracy 26.4281
                Validation Loss: 0.0552 Validation Accuracy: 27.7687
Epoch 4/30, Train Loss: 0.0551 Train Accuracy 27.0352
                Validation Loss: 0.0556 Validation Accuracy: 27.1186
Epoch 5/30, Train Loss: 0.0549 Train Accuracy 27.3985
                Validation Loss: 0.0551 Validation Accuracy: 28.8832
Epoch 6/30, Train Loss: 0.0546 Train Accuracy 28.5131
                Validation Loss: 0.055 Validation Accuracy: 28.6975


KeyboardInterrupt: 

In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = ViTPositional().to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="overfitting_small_ds_positional_numlayer3",
    config = {
    "project": "ml_hw_04",
    "model": "SimpleViTPositional",
    "input": {
        "subset_size": 100,
        "dataset": "train_ds"
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "overfit_small_dataset",
            "group": "ViT"
        }
    }
)
model = train_loop(small_ds, small_ds, "ViT", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


/tmp/ipykernel_1390/3671419950.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


Epoch 1/100, Train Loss: 0.2432 Train Accuracy 19.0
                Validation Loss: 0.2309 Validation Accuracy: 26.0
Epoch 2/100, Train Loss: 0.2344 Train Accuracy 23.0
                Validation Loss: 0.2272 Validation Accuracy: 27.0
Epoch 3/100, Train Loss: 0.2328 Train Accuracy 26.0
                Validation Loss: 0.228 Validation Accuracy: 27.0
Epoch 4/100, Train Loss: 0.2358 Train Accuracy 14.0
                Validation Loss: 0.225 Validation Accuracy: 28.0
Epoch 5/100, Train Loss: 0.2412 Train Accuracy 22.0
                Validation Loss: 0.2292 Validation Accuracy: 28.0
Epoch 6/100, Train Loss: 0.2408 Train Accuracy 11.0
                Validation Loss: 0.2274 Validation Accuracy: 18.0
Epoch 7/100, Train Loss: 0.2335 Train Accuracy 24.0
                Validation Loss: 0.2265 Validation Accuracy: 30.0
Epoch 8/100, Train Loss: 0.2235 Train Accuracy 29.0
                Validation Loss: 0.2211 Validation Accuracy: 27.0
Epoch 9/100, Train Loss: 0.2247 Train Accuracy 27.0
      

epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
train_acc,▁▂▁▂▂▂▂▂▃▂▄▄▄▄▄▅▅▆▇▇▇▇██████████████████
train_loss,███▇█▇▇▅▅▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▂▁▂▂▂▂▂▃▃▃▄▄▅▆▆▇▇▇▇█████████████████████
val_loss,█████▇▇▇▇▆▆▆▅▅▅▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,99
train_acc,100
train_loss,0.00078
val_acc,100
val_loss,0.0008


In [ ]:
from torch.utils.data import Subset

model = ViTPositional().to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="positional_numlayer3_data_augment_and_dropout",
    config = {
    "project": "ml_hw_04",
    "model": "ViTPositional",

    "training": {
        "epochs": 30,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "ViT"
        }
    }
)
model = train_loop(train_ds, val_ds, "ViT", 0.0001, 32, model, 30)
torch.save(model, "positional_vit.pt")
wandb.finish()


/tmp/ipykernel_1390/3741030670.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▁▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇██
train_loss,██▇▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▁▁
val_acc,▁▃▅▆▆▇▇▇▇██████▇█▇█▇
val_loss,█▇▅▃▃▂▂▁▂▁▁▁▁▂▃▄▄▅▇█
epoch,19
train_acc,61.3256
train_loss,0.0319
val_acc,39.2849
val_loss,0.0555


Epoch 1/30, Train Loss: 0.0564 Train Accuracy 24.1143
                Validation Loss: 0.0565 Validation Accuracy: 25.5863
Epoch 2/30, Train Loss: 0.0557 Train Accuracy 25.5076
                Validation Loss: 0.0553 Validation Accuracy: 28.3492
Epoch 3/30, Train Loss: 0.0552 Train Accuracy 26.9606
                Validation Loss: 0.0546 Validation Accuracy: 28.7671
Epoch 4/30, Train Loss: 0.0545 Train Accuracy 28.5579
                Validation Loss: 0.0532 Validation Accuracy: 31.9248
Epoch 5/30, Train Loss: 0.0538 Train Accuracy 30.2896
                Validation Loss: 0.0524 Validation Accuracy: 33.9912
Epoch 6/30, Train Loss: 0.0531 Train Accuracy 31.4789
                Validation Loss: 0.0515 Validation Accuracy: 34.7574
Epoch 7/30, Train Loss: 0.0523 Train Accuracy 33.4644
                Validation Loss: 0.0512 Validation Accuracy: 35.9647
Epoch 8/30, Train Loss: 0.0517 Train Accuracy 34.4098
                Validation Loss: 0.05 Validation Accuracy: 37.6364
Epoch 9/30, Train 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▂▂▃▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████
train_loss,██▇▇▆▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
val_acc,▁▂▂▃▄▄▅▆▆▆▆▆▆▆▆▇▇▇▇▇███▇▇▇████
val_loss,█▇▇▆▅▅▅▄▅▅▃▃▃▄▃▄▂▂▂▃▂▂▂▂▂▂▁▁▁▂
epoch,29
train_acc,42.705
train_loss,0.04604
val_acc,43.9749
val_loss,0.0462


In [ ]:
from torch.utils.data import Subset

model = ViTPositional(n_embd=128, dropout=0.1, num_layers = 3, n_head=4).to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="positional_numlayer3_data_augment_and_dropout",
    config = {
    "project": "ml_hw_04",
    "model": "ViTPositional",

    "training": {
        "epochs": 30,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam",
        "n_head": 4,
        "num_layers": 3,
        "n_embd": 128,
        "dropout": 0.1
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "ViT"
        }
    }
)
model = train_loop(train_ds, val_ds, "ViT", 0.0001, 32, model, 30)
torch.save(model, "positional_vit.pt")
wandb.finish()


Epoch 1/30, Train Loss: 0.0562 Train Accuracy 24.8059
                Validation Loss: 0.0566 Validation Accuracy: 23.9842
Epoch 2/30, Train Loss: 0.056 Train Accuracy 25.2239
                Validation Loss: 0.056 Validation Accuracy: 26.0506
Epoch 3/30, Train Loss: 0.0558 Train Accuracy 25.7166
                Validation Loss: 0.0559 Validation Accuracy: 26.0971
Epoch 4/30, Train Loss: 0.0555 Train Accuracy 25.9504
                Validation Loss: 0.0552 Validation Accuracy: 27.4437
Epoch 5/30, Train Loss: 0.0544 Train Accuracy 29.0854
                Validation Loss: 0.0539 Validation Accuracy: 31.3908
Epoch 6/30, Train Loss: 0.0534 Train Accuracy 31.2898
                Validation Loss: 0.0521 Validation Accuracy: 35.0592
Epoch 7/30, Train Loss: 0.0522 Train Accuracy 33.8077
                Validation Loss: 0.0508 Validation Accuracy: 36.7309
Epoch 8/30, Train Loss: 0.0515 Train Accuracy 34.6835
                Validation Loss: 0.0502 Validation Accuracy: 37.381
Epoch 9/30, Train L